In [4]:
import clustering_metrics as cmetrics
import ploting as cplot
import clustering_external_tools as ctools

import numpy as np
import matplotlib as plt

# Import data:

- X_Data = Rect Ascencion

- Y_Data = Declination

- True_Labels = Original substructure labeling

- Pred_Labels = Substructure labeling according to clustering algorithm

*X_Data, Y_Data, True_Labels, and Pred_Labels must be parallel arrays where the information of a given galaxy must be all in the same index.*
*Any galaxy not considerd as part of a substructur must be marked as -1.*

In [ ]:
#Modify to import your data

X_data = []
Y_data = []

true_labels = []
pred_labels = []

# Plotting the data

**True Data**

In [ ]:
fig, ax = cplot.plot_clusters(X_data, Y_data, true_labels)
plt.show()

**Predicted Data**

In [ ]:
fig, ax = cplot.plot_clusters(X_data, Y_data, pred_labels)
plt.show()

# Prediction Metrics

Completeness, Purity, F1_score, and Rand Index based only in if the galaxy was assigned to a substructure or not

In [ ]:
lax_true_labels = np.where(true_labels >= 0, 1, -1)
lax_pred_labels = np.where(pred_labels >= 0, 1, -1)

tp, fp, tn, fn = cmetrics.get_match(lax_true_labels, lax_pred_labels)

completeness = cmetrics.get_completitud(tp, fn)
purity = cmetrics.get_purity(tp, fn)
f1_score = cmetrics.get_f1_score(completeness, purity)
rand_index = cmetrics.get_rand_index(tp, fp, tn, fn)

In [ ]:
print(f"Completitud: {completeness}")
print(f"Purity: {purity}")
print(f"F1 Score: {f1_score}")
print(f"Rand Index: {rand_index}")

# Substructure Caracteristic Density

*We define substructure caracteristic density based on centroid and 1 sigma deviation of the position of its members*

In [ ]:
np.set_printoptions(precision=6, suppress=True)
data = np.column_stack((X_data, Y_data))

In [ ]:
true_centers = ctools.get_cluster_center(data, true_labels)
true_mean_centers = np.array([cluster['location'] for cluster in true_centers.values()])
true_radial_std = np.array([np.linalg.norm(cluster['scale']) for cluster in true_centers.values()])

In [ ]:
print("TC: -> True cluster centers")
print("TS: -> 1 sigma deviation")

print(f"TC: {true_mean_centers}")
print(f"TS: {true_radial_std}")

In [ ]:
pred_centers = ctools.get_cluster_center(data, pred_labels)
pred_mean_centers = np.array([cluster['location'] for cluster in pred_centers.values()])
pred_radial_std = np.array([np.linalg.norm(cluster['scale']) for cluster in pred_centers.values()])

In [ ]:
print("PC: -> Predicted cluster centers")
print("PS: -> 1 sigma deviation")

print(f"PC: {pred_mean_centers}")
print(f"PS: {pred_radial_std}")

**Plot**


*Predicted clusters are blue. Prdicted cluster centers are marked with a blue dot*

*True clusters are red. True cluster centers are marked with a red X*

*Circles of the corresponding colors mark 1 sigma deviation the from the centroid*

In [ ]:
fig, ax = cplot.plot_comparative_clusters(X_data, Y_data, true_labels, pred_labels)
plt.show()

**Cluster center distances**

In [ ]:
distances = ctools.get_distance_matrix(data, true_labels, pred_labels)

In [ ]:
distances_df = ctools.matrix_as_table(distances)
print("Cluster-Cluster Distances")
print(distances_df)

**Cluster center distances normalized**

Normalization by True

In [ ]:
true_std_expanded = true_radial_std[:, np.newaxis] 
distances_by_true = (distances / true_std_expanded) 

In [ ]:
dist_true_df = ctools.matrix_as_table(distances_by_true)
print("Cluster-Cluster Distances Evaluated by True Clusters deviation")
print(dist_true_df)

Normalization by Predicted

In [ ]:
pred_std_expanded = pred_radial_std[np.newaxis, :] 
distances_by_pred = (distances / pred_std_expanded) 

In [ ]:
dist_pred_df = ctools.matrix_as_table(distances_by_pred)
print("Cluster-Cluster Distances Evaluated by Predicted Clusters deviation")
print(dist_pred_df)

**Cluster 1-Sigma based Areas**

In [ ]:
true_areas = np.pi * (true_radial_std ** 2)

In [ ]:
print(f"True clusters areas: {true_areas}")

In [ ]:
pred_areas = np.pi * (pred_radial_std ** 2)

In [ ]:
print(f"Predicted clusters areas: {pred_areas}")

**Cluster-Cluster Overlapping Areas**

In [ ]:
overlapping_areas = ctools.get_overlapping_areas(distances, true_radial_std, pred_radial_std)

In [ ]:
overlapping_areas_df = ctools.matrix_as_table(overlapping_areas)
print("Cluster-Cluster area overlap")
print(overlapping_areas_df)

**Cluster-Cluster Overlapping Areas Normalized**

Normalization by True

*Amount of true cluster area captured by overlap of true-predicted areas can be used as completeness metric*

In [ ]:
true_areas_expanded = true_areas[:, np.newaxis] 
overlap_by_true = (overlapping_areas / true_areas_expanded) 

In [ ]:
overlap_by_true_df = ctools.matrix_as_table(overlap_by_true)
print("Completeness: Cluster-Cluster area overlap normalized by true cluster area")
print(overlap_by_true_df)

Normalization by True

*Amount of predicted cluster area captured by overlap of true-predicted areas can be used as purity metric*

In [ ]:
pred_areas_expanded = pred_areas[np.newaxis, :] 
overlap_by_pred = (overlapping_areas / pred_areas_expanded)

In [ ]:
overlap_by_pred_df = ctools.matrix_as_table(overlap_by_pred)
print("Purity: Cluster-Cluster area overlap normalized by predicted cluster area")
print(overlap_by_pred_df)

F1 score matrix

*Using cluster area overlap evaluated in true cluster area as completeness and cluster area overlap evaluated in predicted cluster area as purity it is posible to calculate a f1 scopre matrix* 

In [ ]:
f1_matrix = cmetrics.get_f1_score(overlap_by_true, overlap_by_pred)

In [ ]:
f1_matrix_df = ctools.matrix_as_table(f1_matrix)
print("F1 score matrix")
print(f1_matrix)